# Retail Sales — Plotly: Complete Beginner Graphs + Data Analysis

**22 runnable graphs + 12 worked pandas analyses.** Shared reproducible 2,400-row synthetic retail transactions (2025). Each graph is named and explained with purpose, code reading guidance, interpretation, cautions and business action. No object-oriented plotting syntax is required. Run all cells in order.

**Note:** This is a simulated, realistic retail scenario—not actual company sales.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
products = {
    'Electronics': [('Headphones', 2400, .68), ('Smartwatch', 5500, .70), ('Speaker', 3200, .66), ('Tablet', 18500, .78)],
    'Fashion': [('T-Shirt', 899, .50), ('Jeans', 1899, .53), ('Jacket', 3499, .56), ('Shoes', 2599, .58)],
    'Home & Kitchen': [('Cookware', 2799, .59), ('Lamp', 1299, .55), ('Mixer', 4499, .70), ('Bedsheet', 1599, .51)],
    'Beauty': [('Skincare Kit', 1499, .49), ('Perfume', 2999, .54), ('Hair Dryer', 2199, .61), ('Face Wash', 599, .46)]
}
records = []
for i in range(2400):
    month = int(rng.choice(np.arange(1, 13), p=[.065,.065,.075,.07,.075,.075,.08,.08,.09,.11,.12,.095]))
    day = int(rng.integers(1, pd.Period(f'2025-{month:02d}').days_in_month + 1))
    category = str(rng.choice(list(products), p=[.32,.29,.25,.14]))
    product, price, cost_ratio = products[category][int(rng.integers(0, 4))]
    region = str(rng.choice(['North','South','East','West'], p=[.25,.34,.18,.23]))
    channel = str(rng.choice(['Online','Retail Store','Marketplace'], p=[.47,.31,.22]))
    units = int(rng.choice([1,2,3,4,5,6], p=[.46,.27,.14,.07,.04,.02]))
    discount = float(rng.choice([0,.05,.10,.15,.20,.25], p=[.25,.24,.20,.16,.10,.05]))
    revenue = round(units * price * (1 - discount), 2)
    cost = round(units * price * cost_ratio, 2)
    profit = round(revenue - cost, 2)
    delivery = int(max(1, round(rng.normal(3.8 if channel != 'Retail Store' else 1.5, 1.7))))
    rating = float(np.clip(round(rng.normal(4.1 - .07 * max(delivery-4, 0), .55), 1), 1, 5))
    records.append([f'ORD-{i+1:05d}', f'2025-{month:02d}-{day:02d}', region, category, product,
                    str(rng.choice(['Consumer','Corporate','Small Business'], p=[.64,.23,.13])),
                    channel, units, price, discount, revenue, cost, profit,
                    round(100 * profit/revenue, 2), delivery, rating])
columns = ['Order_ID','Order_Date','Region','Category','Product','Customer_Segment','Sales_Channel',
           'Units','Unit_Price','Discount_Pct','Revenue','Cost','Profit','Margin_Pct','Delivery_Days','Customer_Rating']
df = pd.DataFrame(records, columns=columns).sort_values('Order_Date').reset_index(drop=True)
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Month_Num'] = df['Order_Date'].dt.month
df['Month'] = df['Order_Date'].dt.strftime('%b')
df['Quarter'] = 'Q' + df['Order_Date'].dt.quarter.astype(str)
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print('Dataset shape:', df.shape)
print('Total revenue: ₹{:,.0f} | Total profit: ₹{:,.0f}'.format(df['Revenue'].sum(), df['Profit'].sum()))
df.head()


Dataset shape: (2400, 19)
Total revenue: ₹18,118,715 | Total profit: ₹4,466,321


,Order_ID,Order_Date,Region,Category,Product,Customer_Segment,Sales_Channel,Units,Unit_Price,Discount_Pct,Revenue,Cost,Profit,Margin_Pct,Delivery_Days,Customer_Rating,Month_Num,Month,Quarter
0,ORD-00190,2025-01-01,North,Fashion,T-Shirt,Consumer,Marketplace,1,899,0.1,809.1,449.50,359.60,44.44,5,3.4,1,Jan,Q1
1,ORD-00326,2025-01-01,North,Home & Kitchen,Cookware,Small Business,Retail Store,1,2799,0.0,2799.0,1651.41,1147.59,41.00,1,4.1,1,Jan,Q1
2,ORD-00432,2025-01-01,West,Home & Kitchen,Mixer,Consumer,Online,4,4499,0.0,17996.0,12597.20,5398.80,30.00,6,4.6,1,Jan,Q1
3,ORD-00836,2025-01-01,East,Fashion,Jacket,Corporate,Marketplace,3,3499,0.1,9447.3,5878.32,3568.98,37.78,4,3.2,1,Jan,Q1
4,ORD-00972,2025-01-01,South,Electronics,Speaker,Consumer,Retail Store,1,3200,0.1,2880.0,2112.00,768.00,26.67,1,4.6,1,Jan,Q1


## How to read a business chart

**Business context:** You are a retail data analyst studying a fictional Indian retail business with 2,400 transactions in 2025. All values are generated examples, not observed real-world sales.

**Start with a question → choose a measure → group the data → plot → interpret → take action.** For example, “Which region is most profitable?” requires `groupby('Region')['Profit'].sum()`, not a count of records.

**Important measures:** `Revenue` is money received after discounts, `Cost` is estimated associated cost, `Profit = Revenue - Cost`, and `Margin_Pct = Profit / Revenue × 100` for each order. Overall business margin must be calculated as **total profit / total revenue**, not the simple mean of order margins.

**Graph-reading checklist:** (1) Identify the x and y variables and units. (2) Check whether the y-value is a sum, count, mean, or proportion. (3) Locate peaks, lows, exceptions and gaps. (4) Consider sample size, discounts and product mix. (5) Translate the pattern into a testable business action.

**Caution:** Correlation does not establish causation. A category with the highest sales is not automatically the most profitable. Charts from simulated data illustrate analysis techniques, not real market trends.


## Dataset overview and business questions

This sample mimics a multi-category Indian retailer selling through Online, Retail Store and Marketplace channels. **Revenue** is sales after discounts; **Profit = Revenue − Cost**. Values are in Indian rupees (₹).

- Orders: **2,400**; annual revenue: **₹18,118,715**; annual profit: **₹4,466,321**.
- Top category by revenue: **Electronics** (₹11,460,200).
- Highest-revenue month: **November** (₹2,543,583).
- Leading channel by revenue: **Online** (48.3% of total).

> Learning note: These findings describe the **generated practice dataset**, not an actual retailer. A graph shows associations and patterns; it does not prove causation.

In [3]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

## 01. Interactive line chart — monthly revenue

**Purpose:** Explore revenue across the year with hover values.

In [4]:
monthly = df.groupby('Month_Num', as_index=False)['Revenue'].sum()
monthly['Month'] = [month_order[i-1] for i in monthly['Month_Num']]
fig = px.line(monthly, x='Month', y='Revenue', markers=True, title='Monthly Sales Revenue', category_orders={'Month':month_order})
fig.show()

In [5]:
monthly =  df.groupby('Month_Num', as_index=False)['Revenue'].sum()
monthly['Month'] = [month_order[i-1] for i in monthly['Month_Num']]
fig = px.line(monthly, x = 'Month', y = 'Revenue',markers= True, title = 'Month Sales Revenue', category_orders={'Month': month_order})
fig.show()

**Outcome:** **November** records the highest monthly revenue.

**Suggested business decision:** Use hover labels for monthly management reviews.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 02. Interactive bar chart — category revenue

**Purpose:** Rank categories by total revenue.

In [ ]:
totals = df.groupby('Category', as_index=False)['Revenue'].sum().sort_values('Revenue', ascending=False)
fig = px.bar(totals, x='Category', y='Revenue', color='Category', title='Revenue by Product Category')
fig.show()

In [6]:
totals = df.groupby('Category', as_index =False)['Revenue'].sum().sort_values('Revenue', ascending=False)
fig = px.bar(totals, x = 'Category', y= 'Revenue', color='Category', title= 'Revenue by Product Category')
fig.show()

**Outcome:** **Electronics** leads category revenue.

**Suggested business decision:** Identify categories requiring better stock planning.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 03. Donut chart — revenue by sales channel

**Purpose:** Show channel revenue proportions.

In [ ]:
totals = df.groupby('Sales_Channel', as_index=False)['Revenue'].sum()
fig = px.pie(totals, names='Sales_Channel', values='Revenue', hole=.48, title='Sales Channel Revenue Share')
fig.show()

In [7]:
totals = df.groupby('Sales_Channel', as_index=False)['Revenue'].sum()
fig = px.pie(totals, names = 'Sales_Channel', values='Revenue', hole=.48, title= 'Sales Channel Revenue Share')
fig.show()

**Outcome:** **Online** accounts for about **48.3%** of sales.

**Suggested business decision:** Compare marketing allocation with channel revenue share.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 04. Histogram — order values

**Purpose:** Explore common order-value ranges.

In [ ]:
fig = px.histogram(df, x='Revenue', nbins=30, title='Order Value Distribution', labels={'Revenue':'Order Revenue (₹)'})
fig.show()

In [8]:
fig = px.histogram(df, x = 'Revenue', nbins= 30, title= 'Order Value Distribution', labels = {'Revenue' : 'Order Revenue'})
fig.show()

**Outcome:** Median order revenue: **₹3,897**.

**Suggested business decision:** Assess realistic order-value thresholds for promotions.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 05. Bubble scatter — discount, profit and order size

**Purpose:** Compare discount, profit, units and category in one chart.

In [ ]:
fig = px.scatter(df.sample(500, random_state=42), x='Discount_Pct', y='Profit', color='Category', size='Units',
                 hover_data=['Product','Region'], title='Discount vs Profit (Bubble Size = Units)')
fig.show()

In [10]:
fig = px.scatter(df.sample(500, random_state=42), x = 'Discount_Pct', y ='Profit', color='Category', size = 'Units', hover_data = ['Product', 'Region'], title = 'Discount vs Profit (Bubble Size = Units)')
fig.show()

**Outcome:** Profit differs at the same discount depending on price, cost ratio and units.

**Suggested business decision:** Analyse margins by SKU instead of discount alone.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 06. Box plot — delivery times by channel

**Purpose:** Compare medians, spread and long deliveries.

In [ ]:
fig = px.box(df, x='Sales_Channel', y='Delivery_Days', color='Sales_Channel', title='Delivery Days by Channel')
fig.show()

In [12]:
fig = px.box(df, x = 'Sales_Channel', y = 'Delivery_Days', color = 'Sales_Channel', title ='Delivery Days by Channel')
fig.show()

**Outcome:** Retail Store generally has faster fulfilment in this generated data.

**Suggested business decision:** Review late-delivery outliers per channel.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 07. Violin plot — profit by category

**Purpose:** Compare distributions of per-order profit.

In [ ]:
fig = px.violin(df, x='Category', y='Profit', color='Category', box=True, title='Profit Distribution by Category')
fig.show()

In [13]:
fig = px.violin(df, x = 'Category', y='Profit', color = 'Category', box =True, title='Profit Distribution by Category')
fig.show()

**Outcome:** **Electronics** has the highest median profit per order.

**Suggested business decision:** Combine profit-per-order with order counts for decisions.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 08. Area chart — cumulative revenue

**Purpose:** Monitor sales accumulated over time.

In [ ]:
monthly = df.groupby('Month_Num', as_index=False)['Revenue'].sum()
monthly['Cumulative_Revenue'] = monthly['Revenue'].cumsum()
monthly['Month'] = [month_order[i-1] for i in monthly['Month_Num']]
fig = px.area(monthly, x='Month', y='Cumulative_Revenue', title='Cumulative Revenue in 2025', category_orders={'Month':month_order})
fig.show()

In [15]:
monthly = df.groupby('Month_Num', as_index=False)['Revenue'].sum()
monthly['Cumulative_Revenue'] = monthly['Revenue'].cumsum()
monthly['Month'] = [month_order[i - 1] for i in monthly['Month_Num']]
fig = px.area(monthly, x = 'Month', y = 'Cumulative_Revenue', title = 'Cumulative Revenue in 2025', category_orders={'Month':month_order})
fig.show()

**Outcome:** Year-end cumulative revenue is **₹18,118,715**.

**Suggested business decision:** Use running totals for progress against annual goals.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 09. Heatmap — category by month

**Purpose:** Find category-season combinations.

In [ ]:
pivot = df.pivot_table(index='Category', columns='Month_Num', values='Revenue', aggfunc='sum', fill_value=0)
pivot.columns = month_order
fig = px.imshow(pivot, labels={'x':'Month','y':'Category','color':'Revenue (₹)'},
                color_continuous_scale='YlOrRd', title='Category × Month Revenue')
fig.show()

In [19]:
pivot = df.pivot_table(index ='Category', columns='Month_Num', values = 'Revenue', aggfunc= 'sum', fill_value=0)
pivot.columns = month_order
fig = px.imshow(pivot, labels ={'x' : 'Month', 'y' : 'Category', 'color':'Revenue'},
                color_continuous_scale='YlOrRd', title = 'Category x Month Revenue')
fig.show()

**Outcome:** Higher-intensity cells indicate higher revenue in that month-category pair.

**Suggested business decision:** Use high-season observations to plan campaigns.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 10. Treemap — category and product sales

**Purpose:** See revenue contribution within a hierarchy.

In [ ]:
totals = df.groupby(['Category','Product'], as_index=False)['Revenue'].sum()
fig = px.treemap(totals, path=['Category','Product'], values='Revenue', title='Product Revenue Treemap')
fig.show()

In [20]:
totals = df.groupby(['Category','Product'], as_index =False)['Revenue'].sum()
fig = px.treemap(totals, path =['Category','Product'], values='Revenue', title = 'Product Revenue Treemap' )
fig.show()

**Outcome:** Large rectangles represent products with high revenue within a category.

**Suggested business decision:** Prioritize assortment using product-level sales contribution.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 11. Sunburst chart — channel to category

**Purpose:** Explore sales in a multi-level hierarchy.

In [21]:
totals = df.groupby(['Sales_Channel','Category'], as_index=False)['Revenue'].sum()
fig = px.sunburst(totals, path=['Sales_Channel','Category'], values='Revenue', title='Revenue by Channel and Category')
fig.show()

**Outcome:** Each inner segment is a sales channel and outer segments represent its category composition.

**Suggested business decision:** Identify which categories dominate each channel.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## 12. Grouped bar chart — quarterly revenue by region

**Purpose:** Compare regional performance within a quarter.

In [22]:
totals = df.groupby(['Quarter','Region'], as_index=False)['Revenue'].sum()
fig = px.bar(totals, x='Quarter', y='Revenue', color='Region', barmode='group',
             category_orders={'Quarter':['Q1','Q2','Q3','Q4']}, title='Quarterly Revenue by Region')
fig.show()

**Outcome:** **Q4** generates the most revenue overall.

**Suggested business decision:** Review quarterly region contributions to sales growth.

**How to interpret:** Read the chart title and units first, then identify the strongest/weakest group or main trend. Check whether the chart uses total revenue, average per order, order counts, percentages or time; these cannot be compared interchangeably.

**Analysis practice:** Identify the highest and lowest values, calculate the gap or percentage difference using pandas, and explain whether the trend may reflect order volume, product prices, seasonality or mix.

**Limitations:** These are generated transactions. Do not infer causation or conclude that a pattern will hold in other years or real stores without more data.

**Manager question:** What should the retail team monitor next, and what additional data would validate the decision?

## Part B — Additional basic and practical graphs

These ten charts add common beginner-friendly and industry-useful chart families. For each, learn when to use it, what the plotted variable represents, and which business action it might support.

### 13. Horizontal bar chart — regional profits

**Purpose:** Rank geographical markets using an interactive horizontal view.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [24]:
totals = df.groupby('Region',as_index=False)['Profit'].sum().sort_values('Profit')
fig = px.bar(totals, x='Profit', y='Region', orientation='h', title='Regional Total Profit', labels={'Profit':'Profit (₹)'})
fig.show()

**Interpretation:** Sorting makes the largest and smallest profit regions immediately visible.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

### 14. Scatter plot — delivery time and rating

**Purpose:** Investigate satisfaction differences across delivery speed.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [25]:
fig = px.scatter(df, x='Delivery_Days', y='Customer_Rating', color='Sales_Channel', opacity=.4, title='Delivery Days vs Customer Rating')
fig.show()

**Interpretation:** Hover individual orders, but avoid assuming longer delivery causes lower ratings without controlling for confounders.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

### 15. Line chart with markers — monthly profit

**Purpose:** Compare profitable and weaker months.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [26]:
totals = df.groupby('Month_Num',as_index=False)['Profit'].sum()
fig = px.line(totals, x='Month_Num', y='Profit', markers=True, title='Monthly Profit', labels={'Month_Num':'Month','Profit':'Profit (₹)'})
fig.show()

**Interpretation:** Compare the profit trend with monthly revenue rather than assuming both follow identical patterns.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

### 16. Stacked bar chart — channel by category

**Purpose:** Show total revenue as a sum of channel components.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [27]:
totals = df.groupby(['Category','Sales_Channel'],as_index=False)['Revenue'].sum()
fig = px.bar(totals, x='Category', y='Revenue', color='Sales_Channel', barmode='stack', title='Category Revenue by Channel')
fig.show()

**Interpretation:** The height is total category revenue, while colors identify channel contribution.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

### 17. 100% stacked bar chart — channel mix

**Purpose:** Compare composition without category-size differences.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [ ]:
totals = df.groupby(['Category','Sales_Channel'],as_index=False)['Revenue'].sum()
totals['Share_Pct'] = 100 * totals['Revenue'] / totals.groupby('Category')['Revenue'].transform('sum')
fig = px.bar(totals, x='Category', y='Share_Pct', color='Sales_Channel', title='Channel Contribution Within Categories (%)', labels={'Share_Pct':'Revenue share (%)'})
fig.show()

**Interpretation:** Every bar sums to 100%; this hides absolute sales volume, so compare it with a normal stacked chart.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

### 18. ECDF line — revenue distribution

**Purpose:** Read percentiles and threshold coverage interactively.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [28]:
fig = px.ecdf(df, x='Revenue', color='Sales_Channel', title='Order Revenue Cumulative Distribution')
fig.show()

**Interpretation:** Use the cumulative distribution to compare how often each channel generates orders below a target amount.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

### 19. Strip chart — customer rating by region

**Purpose:** Inspect individual observations for each region.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [ ]:
fig = px.strip(df.sample(350,random_state=7), x='Region', y='Customer_Rating', color='Sales_Channel', title='Ratings by Region (Sample)')
fig.show()

**Interpretation:** Individual points show dispersion; apparent clusters are sensitive to sample sizes.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

### 20. Funnel chart — category contribution

**Purpose:** Show categories ranked by sales as a descending comparison, not a conversion funnel.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [ ]:
totals = df.groupby('Category',as_index=False)['Revenue'].sum().sort_values('Revenue',ascending=False)
fig = px.funnel(totals, x='Revenue', y='Category', title='Category Revenue Ranking (Funnel Style)')
fig.show()

**Interpretation:** This is a sorted comparison only. Do not interpret category rows as sequential customer funnel stages.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

### 21. Polar bar chart — quarterly sales

**Purpose:** Compare four quarterly totals on a circular chart.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [29]:
totals = df.groupby('Quarter',as_index=False)['Revenue'].sum().sort_values('Quarter')
fig = px.bar_polar(totals, r='Revenue', theta='Quarter', title='Quarterly Sales — Polar Bars')
fig.show()

**Interpretation:** Circular charts attract attention but conventional bars are usually easier for precise comparisons.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

### 22. Timeline — sample order fulfilment

**Purpose:** Show each order window from purchase date through delivery.

**What to check before plotting:** Confirm the correct unit, grouping, and any data filtering. A sum measures business size, a mean measures typical value, and a count measures activity.

**Code walkthrough:** The pandas preparation creates the fields to plot. The next lines choose a chart, set a title and readable labels, then display it.

In [30]:
sample = df.sample(15, random_state=8).copy()
sample['Start'] = pd.to_datetime(sample['Order_Date'])
sample['Finish'] = sample['Start'] + pd.to_timedelta(sample['Delivery_Days'], unit='D')
fig = px.timeline(sample, x_start='Start', x_end='Finish', y='Order_ID', color='Sales_Channel', title='Sample Delivery Windows')
fig.update_yaxes(autorange='reversed')
fig.show()

**Interpretation:** Timelines show fulfilment duration, but sampled rows alone cannot establish overall service-level performance.

**Practical business application:** Identify the top and bottom groups, compare performance over time or by sales channel, and check whether the findings warrant an inventory, logistics, pricing or marketing experiment.

**Limitations and common mistakes:** Observe scale, sample size and aggregation. An apparent association is not proof of cause. Use a comparison table to confirm exact values before reporting.

**Student task:** Write two sentences: one describing a visible pattern, and another recommending what you would investigate next.

## Part C — Hands-on retail data analysis with pandas

The next twelve worked examples go beyond plotting. They show how a business analyst checks raw data, calculates KPIs, groups records, examines exceptions, and turns results into recommendations. Run each cell and inspect the output.

### Analysis 01 — Data quality and structure

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [31]:
# Review dataset size, sample rows, data types and missing values
print('Rows, columns:', df.shape)
display(df.head(5))
print('Duplicate order IDs:', df['Order_ID'].duplicated().sum())
print('Missing values by field:')
print(df.isna().sum().to_string())
print('Data types:')
print(df.dtypes.to_string())

Rows, columns: (2400, 19)


,Order_ID,Order_Date,Region,Category,Product,Customer_Segment,Sales_Channel,Units,Unit_Price,Discount_Pct,Revenue,Cost,Profit,Margin_Pct,Delivery_Days,Customer_Rating,Month_Num,Month,Quarter
0,ORD-00190,2025-01-01,North,Fashion,T-Shirt,Consumer,Marketplace,1,899,0.1,809.1,449.50,359.60,44.44,5,3.4,1,Jan,Q1
1,ORD-00326,2025-01-01,North,Home & Kitchen,Cookware,Small Business,Retail Store,1,2799,0.0,2799.0,1651.41,1147.59,41.00,1,4.1,1,Jan,Q1
2,ORD-00432,2025-01-01,West,Home & Kitchen,Mixer,Consumer,Online,4,4499,0.0,17996.0,12597.20,5398.80,30.00,6,4.6,1,Jan,Q1
3,ORD-00836,2025-01-01,East,Fashion,Jacket,Corporate,Marketplace,3,3499,0.1,9447.3,5878.32,3568.98,37.78,4,3.2,1,Jan,Q1
4,ORD-00972,2025-01-01,South,Electronics,Speaker,Consumer,Retail Store,1,3200,0.1,2880.0,2112.00,768.00,26.67,1,4.6,1,Jan,Q1


Duplicate order IDs: 0
Missing values by field:
Order_ID            0
Order_Date          0
Region              0
Category            0
Product             0
Customer_Segment    0
Sales_Channel       0
Units               0
Unit_Price          0
Discount_Pct        0
Revenue             0
Cost                0
Profit              0
Margin_Pct          0
Delivery_Days       0
Customer_Rating     0
Month_Num           0
Month               0
Quarter             0
Data types:
Order_ID                       str
Order_Date          datetime64[us]
Region                         str
Category                       str
Product                        str
Customer_Segment               str
Sales_Channel                  str
Units                        int64
Unit_Price                   int64
Discount_Pct               float64
Revenue                    float64
Cost                       float64
Profit                     float64
Margin_Pct                 float64
Delivery_Days                int

**How to explain the outcome:** Check for duplicated IDs, invalid types and missing values before making a chart. A chart based on incomplete transactions can mislead management.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 02 — Core business KPIs

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
total_revenue = df['Revenue'].sum()
total_profit = df['Profit'].sum()
orders = df['Order_ID'].nunique()
units_sold = df['Units'].sum()
kpis = pd.DataFrame({'Metric': ['Total revenue', 'Total profit', 'Total orders', 'Units sold', 'Average order value', 'Weighted profit margin (%)'], 'Value': [round(total_revenue,2), round(total_profit,2), orders, units_sold, round(total_revenue/orders,2), round(100*total_profit/total_revenue,2)]})
display(kpis)

**How to explain the outcome:** Sales value, profitability, units, order count and order value answer different management questions. Weighted margin uses total revenue as its denominator.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 03 — Category sales, margin and order volume

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
category_summary = df.groupby('Category').agg(Orders=('Order_ID','nunique'), Units=('Units','sum'), Revenue=('Revenue','sum'), Profit=('Profit','sum')).sort_values('Revenue', ascending=False)
category_summary['Margin_Pct'] = (100*category_summary['Profit']/category_summary['Revenue']).round(2)
category_summary['Avg_Order_Value'] = (category_summary['Revenue']/category_summary['Orders']).round(2)
display(category_summary.round(2))

**How to explain the outcome:** Compare revenue rank with margin and profit rank. Higher revenue may reflect premium-priced items, not superior efficiency.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 04 — Monthly growth and seasonality

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
month_summary = df.groupby(['Month_Num','Month'], as_index=False).agg(Revenue=('Revenue','sum'), Profit=('Profit','sum'), Orders=('Order_ID','count')).sort_values('Month_Num')
month_summary['Growth_Pct'] = month_summary['Revenue'].pct_change().mul(100).round(2)
month_summary['Profit_Margin_Pct'] = (100*month_summary['Profit']/month_summary['Revenue']).round(2)
display(month_summary)

**How to explain the outcome:** Month-on-month growth explains changes relative to the prior month. The first month has no prior-month comparison, so its growth is missing.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 05 — Region and channel performance

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
region_channel = df.pivot_table(index='Region', columns='Sales_Channel', values='Revenue', aggfunc='sum', fill_value=0)
region_channel['All_Channels'] = region_channel.sum(axis=1)
display(region_channel.round(2))
print('Top region by revenue:', region_channel['All_Channels'].idxmax())

**How to explain the outcome:** Use cross-tabulation to identify where a channel contributes most. Large revenue alone does not tell you the cost or margin of serving that market.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 06 — Top products and Pareto contribution

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
product_revenue = df.groupby('Product')['Revenue'].sum().sort_values(ascending=False)
product_share = pd.DataFrame({'Revenue': product_revenue})
product_share['Share_Pct'] = (product_share['Revenue']/product_share['Revenue'].sum()*100).round(2)
product_share['Cumulative_Share_Pct'] = product_share['Share_Pct'].cumsum().round(2)
display(product_share.head(10))

**How to explain the outcome:** Use the cumulative share to find whether a few products drive most revenue. The 80/20 rule is a hypothesis to test, not a guaranteed outcome.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 07 — Discount effectiveness and profitability

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
df['Discount_Band'] = pd.cut(df['Discount_Pct'], bins=[-0.001,0,0.05,0.10,0.15,1], labels=['0%','0–5%','5–10%','10–15%','15%+'])
discount_summary = df.groupby('Discount_Band', observed=True).agg(Orders=('Order_ID','count'), Revenue=('Revenue','sum'), Profit=('Profit','sum'))
discount_summary['Weighted_Margin_Pct'] = (100*discount_summary['Profit']/discount_summary['Revenue']).round(2)
display(discount_summary.round(2))

**How to explain the outcome:** Compare discounts with weighted margins and sample sizes. These observational comparisons do not prove discounts caused a sales increase.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 08 — Customer segment and average order value

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
segment_summary = df.groupby('Customer_Segment').agg(Orders=('Order_ID','count'), Revenue=('Revenue','sum'), Avg_Units=('Units','mean'), Avg_Rating=('Customer_Rating','mean'))
segment_summary['Avg_Order_Value'] = segment_summary['Revenue']/segment_summary['Orders']
display(segment_summary.round(2))

**How to explain the outcome:** Segments can produce similar revenue but different basket sizes. Plan promotions only after checking order count and product mix.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 09 — Delivery speed and customer feedback

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
df['Delivery_Group'] = pd.cut(df['Delivery_Days'], bins=[-1,2,4,7,100], labels=['0–2 days','3–4 days','5–7 days','8+ days'])
delivery_summary = df.groupby('Delivery_Group', observed=True).agg(Orders=('Order_ID','count'), Avg_Rating=('Customer_Rating','mean'), Avg_Revenue=('Revenue','mean'))
display(delivery_summary.round(2))

**How to explain the outcome:** Determine whether longer delivery groups have lower average ratings. This is descriptive, not a controlled causal test.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 10 — Find exceptions and potential anomalies

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
print('Orders with negative profit:', (df['Profit'] < 0).sum())
print('Largest orders by revenue:')
display(df.nlargest(5, 'Revenue')[['Order_ID','Category','Product','Revenue','Profit','Units','Discount_Pct']])
print('Lowest margin orders:')
display(df.nsmallest(5, 'Margin_Pct')[['Order_ID','Product','Revenue','Profit','Margin_Pct']])

**How to explain the outcome:** Investigate outliers before excluding them; unusually large transactions may be real wholesale purchases or data issues.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 11 — Correlation checks

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
numeric_cols = ['Units','Unit_Price','Discount_Pct','Revenue','Profit','Margin_Pct','Delivery_Days','Customer_Rating']
corr_table = df[numeric_cols].corr(numeric_only=True)
display(corr_table.round(2))

**How to explain the outcome:** Correlation ranges from -1 to +1. It describes linear association, not why a variable changes. Some relationships arise mechanically from how synthetic columns were generated.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

### Analysis 12 — Mini case study: business recommendations

**Business question:** What does this analysis tell us about sales operations?

**Method:** Review the pandas operations in the code, then compare the output values.

In [ ]:
best_category = category_summary['Profit'].idxmax()
peak_month_row = month_summary.loc[month_summary['Revenue'].idxmax()]
fastest_channel = df.groupby('Sales_Channel')['Delivery_Days'].median().idxmin()
print('Highest total-profit category:', best_category)
print('Highest sales month:', peak_month_row['Month'])
print('Fastest median delivery channel:', fastest_channel)
print('Suggested validation: compare category stockouts, monthly promotional calendars and channel fulfilment costs before acting.')

**How to explain the outcome:** A strong analyst ends with an action and a validation plan, not a chart alone. Check whether the observed pattern remains after accounting for order counts and mix.

**Follow-up exercise:** Identify one concrete finding from the printed numbers and one additional field you would want from the real retail system.

## Practice assignments

1. Filter the dataset to the **South** region and remake any three charts.
2. Compare **revenue versus profit** and identify why they differ.
3. Replace total revenue with **average order revenue** in one graph.
4. Explore whether higher discounts coincide with lower margins, **without claiming causation**.
5. Add labels, change colors and write two sentences explaining each chart.

**Final reminder:** All data is synthetic; code is intentionally beginner-friendly and uses no `ax` syntax.

## Final student project — Retail sales presentation

1. Calculate revenue, profit, margin, order count and average order value.
2. Pick five different graphs: time trend, category comparison, distribution, relationship and region-by-channel analysis.
3. Under each plot, write: **Question → Metric → Observation → Business decision → Caveat.**
4. Compare the strongest revenue category with the strongest margin category.
5. Explain why high discount orders may have different margins and what extra data is needed for a causal conclusion.
6. Present three prioritized recommendations with numbers from the analysis tables.

**Knowledge check:** When should you use a line plot instead of a bar chart? How do revenue share and order share differ? When does a median describe sales better than a mean? Why must overall margin be weighted? What is misleading about truncated axes or unlabelled normalization?